# Setup

In [ ]:
! pip install openai

import openai

In [ ]:
import re
from openai import OpenAI

client = OpenAI(
    api_key="xxxx",
    base_url="https://api.deepseek.com"
)


def lmp(base_prompt, query, stop_tokens=None, query_kwargs=None):
    full_prompt = f"{base_prompt}\n{query}"

    messages = [
        {"role": "system", "content": "You are an AI that writes Python code. Output only the code, no explanation, no markdown formatting. Use any variable names you like, and write multi-line code if needed."},
        {"role": "user", "content": full_prompt}
    ]

    use_kwargs = {
        'model': 'deepseek-chat',
        'max_tokens': 512,
        'temperature': 0,
    }
    if stop_tokens:
        use_kwargs['stop'] = stop_tokens
    if query_kwargs:
        use_kwargs.update(query_kwargs)

    response = client.chat.completions.create(
        messages=messages,
        **use_kwargs
    )

    raw_response = response.choices[0].message.content.strip()

    # 提取 markdown 代码块
    code_block_pattern = r"```(?:python)?\s*\n(.*?)```"
    matches = re.findall(code_block_pattern, raw_response, re.DOTALL)
    if matches:
        code = matches[0].strip()
    else:
        # 去掉反引号
        code = raw_response.replace("`", "")

    # ------- 新增：智能清洗，不强制特定关键词 -------
    # 策略：保留所有看起来像 Python 代码的行；去除开头部分的自然语言描述。
    lines = code.split('\n')
    cleaned_lines = []
    started = False  # 是否已经进入代码区

    for line in lines:
        stripped = line.strip()
        if not stripped:
            if started:
                cleaned_lines.append(line)  # 保留代码区内的空行
            continue

        # 判断是否为自然语言（以英文单词开头且不包含任何编程符号）
        # 规则：如果行以字母开头，后面跟空格，并且不包含 '=', '(', ')', ':', '.', '[', ']', '{', '}', 'def', 'import', 'for', 'while', 'if', 'else', 'return' 等，则可能是解释。
        is_natural = (re.match(r'^[A-Za-z]+\s', stripped) and
                      not any(c in stripped for c in '=():.[]{}#') and
                      not any(kw in stripped for kw in ['def', 'import', 'for', 'while', 'if', 'else', 'return', 'print']))

        if not started and is_natural:
            # 跳过开头的自然语言行
            continue
        else:
            started = True
            cleaned_lines.append(line)

    code = '\n'.join(cleaned_lines)

    # 如果最终结果为空，回退到原始响应（极少数情况）
    if not code:
        code = raw_response

    print(query)
    print(code)
    return code

# Simple LMPs

## Pure Python

In [ ]:
prompt_pure_python = '''
# Python script
# get the variable a.
ret_val = a
'''.strip()

In [ ]:
_ = lmp(prompt_pure_python, '# find the sum of variables a and b.', ['#'])

# find the sum of variables a and b.
ret_val = a + b


In [ ]:
_ = lmp(prompt_pure_python, '# find the sum of numbers in a list called values.', ['#'])

# find the sum of numbers in a list called values.
a = sum(values)


In [ ]:
_ = lmp(prompt_pure_python, '# find the difference between the max and min numbers in a list called xs.', ['#'])

# find the difference between the max and min numbers in a list called xs.
ret_val = max(xs) - min(xs)


In [ ]:
_ = lmp(prompt_pure_python, '# see if any number is divisible by 3 in a list called xs.', ['#'])

# see if any number is divisible by 3 in a list called xs.
ret_val = any(x % 3 == 0 for x in xs)


## Using Context

In [ ]:
prompt_context = '''
objects = ['green block', 'green bowl', 'yellow block', 'yellow bowl']
# the yellow block.
ret_val = 'yellow block'
# the blocks.
ret_val = ['green block', 'yellow block']
'''.strip()

In [ ]:
context = "objects = ['blue bowl', 'red block', 'red bowl', 'blue block']"
query = '# the bowls.'

print(context)
_ = lmp(f'{prompt_context}\n{context}', query, ['#', 'objects = ['])

objects = ['blue bowl', 'red block', 'red bowl', 'blue block']
# the bowls.
bowls = [obj for obj in objects if 'bowl' in obj]


In [ ]:
context = "objects = ['blue bowl', 'red block', 'red bowl', 'blue block']"
query = '# sea-colored block.'

print(context)
_ = lmp(f'{prompt_context}\n{context}', query, ['#', 'objects = ['])

objects = ['blue bowl', 'red block', 'red bowl', 'blue block']
# sea-colored block.
def get_sea_colored_block(objects):
    sea_colors = ['blue', 'green', 'teal', 'cyan']
    for obj in objects:
        parts = obj.split()
        if len(parts) == 2:
            color, item = parts
            if color in sea_colors and item == 'block':
                return obj
    return None


In [ ]:
context = '''
objects = ['blue bowl', 'red block', 'red bowl', 'blue block']
# sea-colored block.
ret_val = 'blue block'
objects = ['blue bowl', 'red block', 'red bowl', 'blue block']
'''.strip()
query = '# the other block except for blue block.'

print(context)
_ = lmp(f'{prompt_context}\n{context}', query, ['#', 'objects = ['])

objects = ['blue bowl', 'red block', 'red bowl', 'blue block']
# sea-colored block.
ret_val = 'blue block'
objects = ['blue bowl', 'red block', 'red bowl', 'blue block']
# the other block except for blue block.
def get_objects(objects, description):
    if description == "the yellow block":
        return "yellow block"
    elif description == "the blocks":
        return [obj for obj in objects if "block" in obj]
    elif description == "sea-colored block":
        return "blue block"
    elif description == "the other block except for blue block":
        return [obj for obj in objects if "block" in obj and obj != "blue block"][0]
    else:
        return None


## Using 3rd Party Libraries

In [ ]:
prompt_3lib = '''
import numpy as np
# move all points in pts_np toward the right.
ret_val = pts_np + [0.3, 0]
# move a pt_np toward the top.
ret_val = pt_np + [0, 0.3]
'''.strip()

In [ ]:
query = '# get the left most point in pts_np.'

_ = lmp(prompt_3lib, query, ['#'])

# get the left most point in pts_np.
import numpy as np


In [ ]:
query = '# get the center of pts_np.'

_ = lmp(prompt_3lib, query, ['#'])

# get the center of pts_np.
import numpy as np


In [ ]:
query = '# the closest point in pts_np to pt_np.'

_ = lmp(prompt_3lib, query, ['#'])

# the closest point in pts_np to pt_np.
import numpy as np

def closest_point(pts_np, pt_np):
    distances = np.linalg.norm(pts_np - pt_np, axis=1)
    min_index = np.argmin(distances)
    return pts_np[min_index]


## Using 1st Party Libraries

In [ ]:
prompt_1lib = '''
from utils import get_pos, put_first_on_second
objects = ['gray block', 'gray bowl']
# put the gray block on the gray bowl.
put_first_on_second('gray block', 'gray bowl')
objects = ['purple block', 'purple bowl']
# move the purple bowl toward the left.
target_pos = get_pos('purple bowl') + [-0.3, 0]
put_first_on_second('purple bowl', target_pos)
'''.strip()

In [ ]:
context = "objects = ['blue bowl', 'red block', 'red bowl', 'blue block']"
query = '# move the red block a bit to the right.'

print(context)
_ = lmp(f'{prompt_1lib}\n{context}', query, ['#', 'objects = ['])

objects = ['blue bowl', 'red block', 'red bowl', 'blue block']
# move the red block a bit to the right.
put_first_on_second('red block', get_pos('red block') + [0.3, 0])


In [ ]:
context = "objects = ['blue bowl', 'red block', 'red bowl', 'blue block']"
query = '# put the blue block on the bowl with the same color.'

print(context)
_ = lmp(f'{prompt_1lib}\n{context}', query, ['#', 'objects = ['])

objects = ['blue bowl', 'red block', 'red bowl', 'blue block']
# put the blue block on the bowl with the same color.
put_first_on_second('blue block', 'blue bowl')


## Combined Examples

In [ ]:
prompt_combined = '''
import numpy as np
from utils import get_pos, put_first_on_second
objects = ['cyan block', 'cyan bowl', 'pink bowl']
# put the cyan block in cyan bowl.
put_first_on_second('cyan block', 'cyan bowl')
objects = ['gray block', 'silver block', 'gray bowl']
# place the top most block on the gray bowl.
names = ['gray block', 'silver block']
positions = np.array([get_pos(name) for name in names])
name = names[np.argmax(positions[:,1])]
put_first_on_second(name, 'gray bowl')
objects = ['purple block', 'purple bowl']
# put the purple bowl to the left of the purple block.
target_pos = get_pos('purple block') + [-0.3, 0]
put_first_on_second('purple bowl', target_pos)
'''.strip()

In [ ]:
context = "objects = ['red block', 'blue bowl', 'blue block', 'red bowl']"
query = '# move the left most bowl toward the right.'

print(context)
_ = lmp(f'{prompt_combined}\n{context}', query, ['#', 'objects = ['])

objects = ['red block', 'blue bowl', 'blue block', 'red bowl']
# move the left most bowl toward the right.
import numpy as np
from utils import get_pos, put_first_on_second

names = ['blue bowl', 'red bowl']
positions = np.array([get_pos(name) for name in names])
leftmost_bowl = names[np.argmin(positions[:, 0])]
target_pos = get_pos(leftmost_bowl) + [0.3, 0]
put_first_on_second(leftmost_bowl, target_pos)


In [ ]:
context = "objects = ['red block', 'blue bowl', 'blue block', 'red bowl']"
query = '# place the blocks in bowls with their colors.'

print(context)
_ = lmp(f'{prompt_combined}\n{context}', query, ['#', 'objects = ['])

objects = ['red block', 'blue bowl', 'blue block', 'red bowl']
# place the blocks in bowls with their colors.
import numpy as np
from utils import get_pos, put_first_on_second


In [ ]:
context = "objects = ['red block', 'blue bowl', 'blue block', 'red bowl']"
query = '# move the red block to the middle of the bowls.'

print(context)
_ = lmp(f'{prompt_combined}\n{context}', query, ['#', 'objects = ['])

objects = ['red block', 'blue bowl', 'blue block', 'red bowl']
# move the red block to the middle of the bowls.
import numpy as np
from utils import get_pos, put_first_on_second


# Complex LMPs

## Control Flows

In [ ]:
prompt_ctrl = '''
from utils import get_pos, put_first_on_second
# move the orange block toward the top.
target_pos = get_pos('orange block') + [0, 0.3]
put_first_on_second('orange block', target_pos)
# move the purple bowl toward the left.
target_pos = get_pos('purple bowl') + [-0.3, 0]
put_first_on_second('purple bowl', target_pos)
'''

In [ ]:
context = "objects = ['red block', 'blue bowl', 'blue block', 'red bowl']"
query = '# while the red block is to the left of the blue bowl, move it to the right 5cm at a time.'

_ = lmp(f'{prompt_ctrl}\n{context}', query, ['#', 'objects = ['])

# while the red block is to the left of the blue bowl, move it to the right 5cm at a time.
while get_pos('red block')[0] < get_pos('blue bowl')[0]:
    target_pos = get_pos('red block') + [0.05, 0]
    put_first_on_second('red block', target_pos)


In [ ]:
context = "objects = ['yellow block', 'green bowl', 'green block', 'yellow bowl']"
query = "# move the yellow block toward its bowl 1cm at a time until their distance is less than 5cm apart."

_ = lmp(f'{prompt_ctrl}\n{context}', query, ['#', 'objects = ['])

# move the yellow block toward its bowl 1cm at a time until their distance is less than 5cm apart.
from utils import get_pos, put_first_on_second
import math

yellow_block = 'yellow block'
yellow_bowl = 'yellow bowl'

while True:
    block_pos = get_pos(yellow_block)
    bowl_pos = get_pos(yellow_bowl)
    dx = bowl_pos[0] - block_pos[0]
    dy = bowl_pos[1] - block_pos[1]
    dist = math.hypot(dx, dy)
    if dist < 0.05:
        break


## Calling Other LMPs

In [ ]:
prompt_modular_ui = '''
import numpy as np
from utils import get_pos, put_first_on_second, parse_obj
objects = ['yellow block', 'yellow bowl', 'gray block', 'gray bowl']
# move the sun colored block toward the left.
block_name = parse_obj('sun colored block')
target_pos = get_pos(block_name) + [-0.3, 0]
put_first_on_second(block_name, target_pos)
objects = ['white block', 'white bowl', 'yellow block', 'yellow bowl']
# place the block closest to the blue bowl on the other bowl.
block_name = parse_obj('the block closest to the blue bowl', f'objects = {objects}')
bowl_name = parse_obj('a bowl other than the blue bowl', f'objects = {objects}')
put_first_on_second(block_name, bowl_name)
'''.strip()

prompt_modular_parse_obj = '''
import numpy as np
from utils import get_pos
objects = ['brown bowl', 'green block', 'brown block', 'green bowl']
# the blocks.
ret_val = ['brown block', 'green block']
# the sky colored block.
ret_val = 'blue block'
objects = ['orange block', 'cyan block', 'purple bowl', 'gray bowl']
# the right most block.
block_names = ['orange block', 'cyan block']
block_positions = np.array([get_pos(block_name) for block_name in block_names])
right_block_name = block_names[np.argmax(block_positions[:, 0])]
ret_val = right_block_name
'''.strip()

In [ ]:
context = "objects = ['red block', 'blue bowl', 'blue block', 'red bowl']"
query = '# while the left most block is the red block, move it toward the right.'

print(context)
_ = lmp(f'{prompt_modular_ui}\n{context}', query, ['#', 'objects = ['])

objects = ['red block', 'blue bowl', 'blue block', 'red bowl']
# while the left most block is the red block, move it toward the right.
block_name = parse_obj('the left most block')
while block_name == 'red block':
    target_pos = get_pos(block_name) + [0.3, 0]
    put_first_on_second(block_name, target_pos)
    block_name = parse_obj('the left most block')


In [ ]:
context = "objects = ['red block', 'blue bowl', 'blue block', 'red bowl']"
query = '# the left most block.'

print(context)
_ = lmp(f'{prompt_modular_parse_obj}\n{context}', query, ['#', 'objects = ['])

objects = ['red block', 'blue bowl', 'blue block', 'red bowl']
# the left most block.
block_names = ['red block', 'blue block']
block_positions = np.array([get_pos(block_name) for block_name in block_names])
left_block_name = block_names[np.argmin(block_positions[:, 0])]
ret_val = left_block_name


## Function-Generating LMPs

In [ ]:
prompt_f_gen = '''
import numpy as np
from utils import get_pos, get_obj_bbox_xyxy
# define function: total = get_total(xs=numbers).
def get_total(xs):
    return np.sum(xs)
'''.strip()

In [ ]:
query = '# define function: get_objs_bigger_than_area_th(obj_names, bbox_area_th).'

_ = lmp(f'{prompt_f_gen}', query, ['# define', '# example'])

# define function: get_objs_bigger_than_area_th(obj_names, bbox_area_th).
import numpy as np
from utils import get_pos, get_obj_bbox_xyxy

def get_total(xs):
    return np.sum(xs)

def get_objs_bigger_than_area_th(obj_names, bbox_area_th):
    objs = []
    for obj_name in obj_names:
        bbox = get_obj_bbox_xyxy(obj_name)
        if bbox is not None:
            x1, y1, x2, y2 = bbox
            area = (x2 - x1) * (y2 - y1)
            if area > bbox_area_th:
                objs.append(obj_name)
    return objs


## Hierarchical Function-Generating LMPs

In [ ]:
query = '# define function: get_obj_bbox_area(obj_name).'

_ = lmp(prompt_f_gen, query, ['# define', '# example'])

# define function: get_obj_bbox_area(obj_name).
import numpy as np
from utils import get_pos, get_obj_bbox_xyxy

def get_total(xs):
    return np.sum(xs)

def get_obj_bbox_area(obj_name):
    bbox = get_obj_bbox_xyxy(obj_name)
    x1, y1, x2, y2 = bbox
    width = x2 - x1
    height = y2 - y1
    area = width * height
    return area


In [ ]:
prompt_hier_f_gen = '''
import numpy as np
from utils import get_obj_pos, get_obj_bbox_xyxy
# define function: total = get_total(xs=numbers).
def get_total(xs):
    return np.sum(xs)
# define function: pt_np = move_pt_left(pt_np, dist).
def move_pt_left(pt_np, dist):
    delta = np.array([-dist, 0])
    return translate_pt_np(pt_np, delta=delta)
'''.strip()

In [ ]:
query = '# define function: get_obj_bbox_area(obj_name).'

_ = lmp(prompt_hier_f_gen, query, ['# define', '# example'])

# define function: get_obj_bbox_area(obj_name).
def get_obj_bbox_area(obj_name):
    bbox = get_obj_bbox_xyxy(obj_name)
    x1, y1, x2, y2 = bbox
    width = x2 - x1
    height = y2 - y1
    area = width * height
    return area


In [ ]:
query = '# define function: area = get_bbox_area(bbox_xyxy).'

_ = lmp(prompt_hier_f_gen, query, ['# define', '# example'])

# define function: area = get_bbox_area(bbox_xyxy).
def get_bbox_area(bbox_xyxy):
    x1, y1, x2, y2 = bbox_xyxy
    return (x2 - x1) * (y2 - y1)


## Combined Example

In [ ]:
prompt_combined_parse_pos = '''
from utils import get_obj_pos

# right of the red block.
ret_val = get_obj_pos('red block') + np.array([0.3, 0])
# a bit below the cyan bowl.
ret_val = get_obj_pos('cyan bowl') + np.array([0, -0.1])
# a point between the cyan block and purple bowl.
pos = get_mid_point_np(pt0=get_obj_pos('cyan block'), pt1=get_obj_pos('purple bowl'))
ret_val = pos
# the corner closest to the yellow block.
corner_positions = get_corner_positions()
closest_corner_idx = get_closest_idx(points=corner_positions, point=get_obj_pos('yellow block'))
closest_corner_pos = corner_positions(closest_corner_idx)
ret_val = closest_corner_pos
'''.strip()

prompt_combined_parse_obj = '''
import numpy as np
from utils import get_obj_pos, get_objs_bigger_than_area_th

objects = ['brown bowl', 'green block', 'brown block', 'green bowl', 'blue bowl', 'blue block']
# the blocks.
ret_val = ['brown block', 'blue block']
# the sky colored block.
ret_val = 'blue block'
objects = ['blue block', 'cyan block', 'purple bowl', 'gray bowl', 'brown bowl', 'pink block']
# the right most block.
block_names = ['cyan block', 'pink block', 'blue block']
block_positions = np.array([get_obj_pos(block_name) for block_name in block_names])
right_block_name = block_names[np.argmax(block_positions[:, 0])]
ret_val = right_block_name
objects = ['blue block', 'cyan block', 'purple bowl', 'brown bowl', 'purple block']
# blocks above the brown bowl.
block_names = ['blue block', 'cyan block', 'purple block']
brown_bowl_pos = get_obj_pos('brown bowl')
use_block_names = [name for name in block_names if get_obj_pos(name)[1] > brown_bowl_pos[1]]
ret_val = use_block_names
'''.strip()

prompt_combined_ui = '''
import numpy as np
from utils import get_obj_pos, put_first_on_second, parse_obj

objects = ['green block', 'green bowl', 'yellow block', 'yellow bowl']
# move the bowls below the yellow block upwards.
bowl_names = parse_obj('the bowls below the yellow block', f'objects = {objects}')
for bowl_name in bowl_names:
  target_pos = get_obj_pos(bowl_name) + np.array([0, 0.1])
  put_first_on_second(bowl_name, target_pos)
objects = ['white block', 'white bowl', 'yellow block', 'yellow bowl']
# if the banana colored bowl is the top most bowl, then place it on the white bowl.
bowl_name = parse_obj('the banana colored bowl', f'objects = {objects}')
top_bowl_name = parse_obj('the top most bowl', f'objects = {objects}')
if bowl_name == top_bowl_name:
  put_first_on_second(bowl_name, 'white bowl')
objects = ['blue block', 'blue bowl', 'red block', 'red bowl']
# place the block closest to the blue bowl on the other bowl.
block_name = parse_obj('the block closest to the blue bowl', f'objects = {objects}')
bowl_name = parse_obj('a bowl other than the blue bowl', f'objects = {objects}')
put_first_on_second(block_name, 'red bowl')
'''.strip()

In [ ]:
context = "objects = ['red block', 'blue bowl', 'blue block', 'red bowl']"
query = '# while there are blocks with area bigger than 0.2 that are left of the red bowl, move them toward the right.'

print(context)
_ = lmp(f'{prompt_combined_ui}\n{context}', query, ['#', 'objects = ['])

objects = ['red block', 'blue bowl', 'blue block', 'red bowl']
# while there are blocks with area bigger than 0.2 that are left of the red bowl, move them toward the right.
block_names = parse_obj('blocks with area bigger than 0.2 that are left of the red bowl', f'objects = {objects}')
while len(block_names) > 0:
  for block_name in block_names:
    target_pos = get_obj_pos(block_name) + np.array([0.1, 0])
    put_first_on_second(block_name, target_pos)
  block_names = parse_obj('blocks with area bigger than 0.2 that are left of the red bowl', f'objects = {objects}')


In [ ]:
context = "objects = ['red block', 'blue bowl', 'blue block', 'red bowl']"
query = '# blocks with area bigger than 0.2 that are left of the red bowl.'

print(context)
_ = lmp(f'{prompt_combined_parse_obj}\n{context}', query, ['#', 'objects = ['])

objects = ['red block', 'blue bowl', 'blue block', 'red bowl']
# blocks with area bigger than 0.2 that are left of the red bowl.
block_names = ['red block', 'blue block']
red_bowl_pos = get_obj_pos('red bowl')
use_block_names = [name for name in block_names if get_obj_pos(name)[0] < red_bowl_pos[0]]
use_block_names = get_objs_bigger_than_area_th(use_block_names, 0.2)
ret_val = use_block_names
